<a href="https://colab.research.google.com/github/sudipto2104/ai-platform-engineering-portfolio/blob/main/01-rag-vector-databases/01_rag_openai_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/sudipto2104/ai-platform-engineering-portfolio.git

fatal: destination path 'ai-platform-engineering-portfolio' already exists and is not an empty directory.


In [2]:
%cd ai-platform-engineering-portfolio/01-rag-vector-databases
!ls

/content/ai-platform-engineering-portfolio/01-rag-vector-databases
ai-platform-engineering-portfolio  ingestion.py  README.md	   ui_gradio.py
chroma_db			   __pycache__	 requirements.txt
docs				   rag_chain.py  run.py


In [3]:
!pip install -q -r requirements.txt

In [4]:
!pip install -q langchain langchain-community langchain-openai chromadb gradio python-dotenv pypdf

In [5]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ OpenAI API Key is set!")

✅ OpenAI API Key is set!


In [6]:
# Update rag_chain.py with OpenAI
with open('rag_chain.py', 'w') as f:
    f.write('''from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# Load OpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="platform_knowledge"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

template = """You are a helpful Platform Engineering Assistant.
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask_question(question: str):
    print(f"\\n❓ Question: {question}")
    response = rag_chain.invoke(question)
    print(f"💡 Answer: {response}")
    return response

print("✅ RAG Chain with OpenAI is ready!")
''')

print("✅ rag_chain.py successfully updated with OpenAI!")

✅ rag_chain.py successfully updated with OpenAI!


In [7]:
ingestion_py_content = '''from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import os

DATA_PATH = "docs"
VECTOR_DB_PATH = "chroma_db"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 80

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def main():
    if os.path.exists(VECTOR_DB_PATH):
        print("Vector database already exists. Loading it...")
        vectorstore = Chroma(
            persist_directory=VECTOR_DB_PATH,
            embedding_function=embeddings,
            collection_name="platform_knowledge"
        )
        print("Vector database loaded.")
        return

    print("Creating new vector database...")
    documents = []
    for filename in os.listdir(DATA_PATH):
        if filename.endswith(".pdf"):
            file_path = os.path.join(DATA_PATH, filename)
            loader = PyPDFLoader(file_path)
            documents.extend(loader.load())

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        add_start_index=True
    )
    all_splits = text_splitter.split_documents(documents)

    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH,
        collection_name="platform_knowledge"
    )
    vectorstore.persist()
    print("Vector database created and persisted.")

if __name__ == "__main__":
    main()
'''

with open('ingestion.py', 'w') as f:
    f.write(ingestion_py_content)
print("✅ ingestion.py successfully updated with OpenAI Embeddings!")



✅ ingestion.py successfully updated with OpenAI Embeddings!


In [8]:
ingestion_py_content_fixed = '''from langchain.document_loaders import PyPDFLoader, TextLoader # Added TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import os

DATA_PATH = "docs"
VECTOR_DB_PATH = "chroma_db"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 80

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def main():
    if os.path.exists(VECTOR_DB_PATH):
        print("Vector database already exists. Loading it...")
        vectorstore = Chroma(
            persist_directory=VECTOR_DB_PATH,
            embedding_function=embeddings,
            collection_name="platform_knowledge"
        )
        print("Vector database loaded.")
        return

    print("Creating new vector database...")
    documents = []
    # Loop through files in DATA_PATH and load them
    for filename in os.listdir(DATA_PATH):
        file_path = os.path.join(DATA_PATH, filename)
        if filename.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            documents.extend(loader.load())
        elif filename.endswith(".txt"): # Added handling for .txt files
            loader = TextLoader(file_path)
            documents.extend(loader.load())
        else:
            print(f"Skipping unknown file type: {filename}") # Informative message


    if not documents: # Added check for empty documents
        print(f"Warning: No documents (PDF or TXT) found in '{DATA_PATH}'. Please ensure there are files to ingest.")
        return # Exit if no documents

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        add_start_index=True
    )
    all_splits = text_splitter.split_documents(documents)

    if not all_splits: # Added check for empty splits
        print("Warning: No text chunks were generated from the loaded documents. This might happen if documents are empty or the splitter configuration is too restrictive.")
        return # Exit if no splits

    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH,
        collection_name="platform_knowledge"
    )
    vectorstore.persist()
    print("Vector database created and persisted.")

if __name__ == "__main__":
    main()
'''

with open('ingestion.py', 'w') as f:
    f.write(ingestion_py_content_fixed)
print("✅ ingestion.py successfully updated with OpenAI Embeddings and .txt loader!")

from ingestion import main as run_ingestion

print("🚀 Starting Ingestion...")
run_ingestion()
print("✅ Ingestion process complete!")

✅ ingestion.py successfully updated with OpenAI Embeddings and .txt loader!
🚀 Starting Ingestion...
Vector database already exists. Loading it...


/content/ingestion.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector database loaded.
✅ Ingestion process complete!


In [9]:
from rag_chain import ask_question

# Test questions
ask_question("What is Kubernetes Network Policy?")
ask_question("Why do platform engineers use Cilium?")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ RAG Chain with OpenAI is ready!

❓ Question: What is Kubernetes Network Policy?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


💡 Answer: Kubernetes Network Policy is a specification that defines how groups of pods can communicate with each other and with other network endpoints. It allows you to control the traffic flow at the IP address or port level, enabling you to restrict or allow traffic based on defined rules. Network Policies can be used to enforce security by isolating pods and controlling access to services, thus enhancing the overall security posture of applications running in a Kubernetes cluster.

❓ Question: Why do platform engineers use Cilium?
💡 Answer: Platform engineers use Cilium for several reasons:

1. **Network Security**: Cilium provides advanced network security features through its use of eBPF (extended Berkeley Packet Filter), allowing for fine-grained control over network traffic and policies.

2. **Performance**: By leveraging eBPF, Cilium can achieve high performance and low latency in networking, as it operates at the kernel level and avoids the overhead of traditional networking 

'Platform engineers use Cilium for several reasons:\n\n1. **Network Security**: Cilium provides advanced network security features through its use of eBPF (extended Berkeley Packet Filter), allowing for fine-grained control over network traffic and policies.\n\n2. **Performance**: By leveraging eBPF, Cilium can achieve high performance and low latency in networking, as it operates at the kernel level and avoids the overhead of traditional networking solutions.\n\n3. **Kubernetes Integration**: Cilium is designed to work seamlessly with Kubernetes, providing capabilities like service mesh, load balancing, and network policy enforcement that are essential for cloud-native applications.\n\n4. **Observability**: Cilium offers powerful observability features, enabling platform engineers to monitor and troubleshoot network traffic and performance issues effectively.\n\n5. **Scalability**: Cilium is built to scale with modern microservices architectures, making it suitable for large and compl

✅ API Key loaded successfully!
Key starts with: sk-proj-GTSxyLCZ4No1...


In [6]:
import sys
import os
from google.colab import userdata # Import userdata here

# Set OpenAI API Key before importing rag_chain
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ OpenAI API Key is set for Gradio app!")

# Explicitly add the directory containing rag_chain.py to sys.path
rag_chain_dir = '/content/ai-platform-engineering-portfolio/01-rag-vector-databases'
if rag_chain_dir not in sys.path:
    sys.path.insert(0, rag_chain_dir) # Use insert(0, ...) to prioritize this path

import gradio as gr
from rag_chain import ask_question

def respond(message, history):
    if not message or message.strip() == "":
        return "Please ask a question."

    response = ask_question(message)
    return response

demo = gr.ChatInterface(
    fn=respond,
    title="Platform Engineering Assistant",
    description="Ask me anything about Kubernetes, Platform Engineering, Cilium, etc.",
    theme="soft"
)

demo.launch(share=True)

✅ OpenAI API Key is set for Gradio app!


/content/ai-platform-engineering-portfolio/01-rag-vector-databases/rag_chain.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ RAG Chain with OpenAI is ready!


/usr/local/lib/python3.12/dist-packages/gradio/components/chatbot.py:229: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ecc78f18f358db3834.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
import gradio as gr
from rag_chain import ask_question

def respond(message, history):
    if not message or message.strip() == "":
        return "Please ask a question."
    response = ask_question(message)
    return response

demo = gr.ChatInterface(
    fn=respond,
    title="🤖 Platform Engineering Assistant",
    description="Ask me anything about Kubernetes, Cilium, ArgoCD, Backstage, or Platform Engineering",
    theme="soft",
    autofocus=True
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/components/chatbot.py:229: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51711dd349e14d36b6.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
